In [1]:
"""
Formal hypothesis tests for H1-H3 (Reviewer 2, Comment 4).

H1: Russian energy companies' influence declines during the war.
H2: Western oil & gas companies' influence rises during the war.
H3: The global energy network becomes less cohesive during the war.

Two complementary tests are run for each hypothesis:

  [1] Pre-war (windows 1-6, Jan 2020-Dec 2021) vs. war (windows 7-12,
      Jan 2022-Dec 2023) -- one-sided Mann-Whitney U test.
  [2] Monotonic trend across all 12 windows -- one-sided Spearman
      rank correlation between window index (time) and the metric.
      This doesn't depend on where the pre/war cutoff is drawn, so
      agreement between [1] and [2] is a robustness check.


"""

import os
import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import mannwhitneyu, spearmanr

RUSSIAN = ['Gazprom', 'Inter_Rao', 'Rosneft']
WESTERN_OG = ['BP', 'ConocoPhillips', 'Chevron', 'Eni', 'Equinor',
              'ExxonMobil', 'Suncor', 'Tullow_Oil']

PRE_WAR = range(1, 7)   # windows 1-6
WAR = range(7, 13)      # windows 7-12


def global_reaching_centrality(G):
    if len(G) == 0:
        return 0.0
    closeness_dict = nx.closeness_centrality(G)
    max_centrality = max(closeness_dict.values())
    grc = sum(max_centrality - c for c in closeness_dict.values()) / (len(G) - 1) if len(G) > 1 else 0
    return grc


def topology_metrics(fr):
    """Same threshold-network construction as the paper: theta = mean(FR),
    directed edge target<-source when FR_ij > theta, diagonal zeroed."""
    theta = fr.values.mean()
    A = (fr > theta).astype(int)
    np.fill_diagonal(A.values, 0)

    G = nx.DiGraph()
    G.add_nodes_from(fr.columns)
    for i in fr.index:
        for j in fr.columns:
            if i != j and A.loc[i, j] == 1:
                G.add_edge(j, i)

    density = nx.density(G)
    clustering = nx.average_clustering(G.to_undirected())

    if len(G) > 0:
        largest_scc = max(nx.strongly_connected_components(G), key=len)
        Gs = G.subgraph(largest_scc)
        aspl = nx.average_shortest_path_length(Gs) if len(Gs) > 1 else np.nan
    else:
        aspl = np.nan

    grc = global_reaching_centrality(G)
    return density, clustering, aspl, grc


def load_data():
    ftot_rows, topo_rows = [], []
    for w in range(1, 13):
        path = f"FMin{w}.csv"
        if not os.path.exists(path):
            raise FileNotFoundError(f"{path} not found in this folder.")
        fr = pd.read_csv(path, index_col=0)
        fr.index = [c.strip() for c in fr.index]
        fr.columns = [c.strip() for c in fr.columns]

        f_tot = fr.sum(axis=0)  # Eq. 8: column sums
        period = "pre_war" if w in PRE_WAR else "war"

        for company in RUSSIAN:
            if company in f_tot.index:
                ftot_rows.append({"window": w, "period": period, "group": "Russian", "F_tot": f_tot[company]})
        for company in WESTERN_OG:
            if company in f_tot.index:
                ftot_rows.append({"window": w, "period": period, "group": "Western_OG", "F_tot": f_tot[company]})

        density, clustering, aspl, grc = topology_metrics(fr)
        topo_rows.append({"window": w, "period": period, "Density": density,
                           "Clustering": clustering, "ASPL": aspl, "GRC": grc})

    return pd.DataFrame(ftot_rows), pd.DataFrame(topo_rows)


def one_sided_trend_p(window, values, expect_positive):
    rho, p_two = spearmanr(window, values)
    matches = (rho > 0) == expect_positive
    p_one = p_two / 2 if matches else 1 - p_two / 2
    return rho, p_one


def test_group(ftot_df, group, alt, expect_trend_positive, label):
    sub = ftot_df[ftot_df.group == group]
    pre = sub[sub.period == "pre_war"].F_tot
    war = sub[sub.period == "war"].F_tot

    U, p_mwu = mannwhitneyu(pre, war, alternative=alt)
    rho, p_trend = one_sided_trend_p(sub.window, sub.F_tot, expect_trend_positive)

    print(f"\n{label}")
    print(f"  pre-war: n={len(pre)}, median={pre.median():.4f}   "
          f"war: n={len(war)}, median={war.median():.4f}")
    print(f"  [1] Mann-Whitney U (pre-war vs war): U={U:.1f}, p={p_mwu:.4f}")
    print(f"  [2] Spearman trend (window vs F_tot): rho={rho:+.3f}, p={p_trend:.4f}")


def test_topology(topo_df):
    directions = {
        "Density": ("greater", False),
        "Clustering": ("greater", False),
        "ASPL": ("less", True),
        "GRC": ("greater", False),
    }
    print("\nH3: network topology metrics (n=6 windows per period)")
    for metric, (alt, expect_trend_positive) in directions.items():
        pre = topo_df[topo_df.period == "pre_war"][metric]
        war = topo_df[topo_df.period == "war"][metric]
        U, p_mwu = mannwhitneyu(pre, war, alternative=alt)
        rho, p_trend = one_sided_trend_p(topo_df.window, topo_df[metric], expect_trend_positive)

        direction = "decrease" if alt == "greater" else "increase"
        print(f"  {metric:<12} pre-war={pre.median():.4f}  war={war.median():.4f}  "
              f"(predicts {direction})")
        print(f"    [1] MWU p={p_mwu:.4f}    [2] trend rho={rho:+.3f}, p={p_trend:.4f}")


def main():
    ftot_df, topo_df = load_data()

    print("=" * 70)
    print("H1/H2: firm-level total influence F_tot (Eq. 8)")
    print("=" * 70)
    test_group(ftot_df, "Russian", "greater", False,
               "H1: Russian firms' F_tot declines (pre-war > war; trend rho < 0 expected)")
    test_group(ftot_df, "Western_OG", "less", True,
               "H2: Western O&G firms' F_tot rises (pre-war < war; trend rho > 0 expected)")

    print("\n" + "=" * 70)
    test_topology(topo_df)

    ftot_df.to_csv("hypothesis_test_ftot.csv", index=False)
    topo_df.to_csv("hypothesis_test_topology.csv", index=False)
    print("\nSaved hypothesis_test_ftot.csv and hypothesis_test_topology.csv")


if __name__ == "__main__":
    main()


C:\Users\Didar\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Didar\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


H1/H2: firm-level total influence F_tot (Eq. 8)

H1: Russian firms' F_tot declines (pre-war > war; trend rho < 0 expected)
  pre-war: n=18, median=1.0120   war: n=18, median=0.9056
  [1] Mann-Whitney U (pre-war vs war): U=221.0, p=0.0321
  [2] Spearman trend (window vs F_tot): rho=-0.360, p=0.0155

H2: Western O&G firms' F_tot rises (pre-war < war; trend rho > 0 expected)
  pre-war: n=48, median=1.0921   war: n=48, median=1.0768
  [1] Mann-Whitney U (pre-war vs war): U=1105.0, p=0.3667
  [2] Spearman trend (window vs F_tot): rho=+0.072, p=0.2426


H3: network topology metrics (n=6 windows per period)
  Density      pre-war=0.5982  war=0.5972  (predicts decrease)
    [1] MWU p=0.5000    [2] trend rho=+0.256, p=0.7892
  Clustering   pre-war=0.8451  war=0.8309  (predicts decrease)
    [1] MWU p=0.0130    [2] trend rho=-0.538, p=0.0354
  ASPL         pre-war=1.4018  war=1.4028  (predicts increase)
    [1] MWU p=0.5000    [2] trend rho=-0.256, p=0.7892
  GRC          pre-war=0.0618  war=0.0